In [1]:
# Installation
!pip install pyspark -q

print("✅ Installation terminée !")

✅ Installation terminée !


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BankTransactionStreaming") \
    .getOrCreate()

print("✅ Spark Session créée !")

✅ Spark Session créée !


In [3]:
import pandas as pd

df = pd.read_csv("/content/PS_20174392719_1491204439457_log.csv")
print(f"✅ Dataset chargé : {len(df)} transactions")
print(df.head())

✅ Dataset chargé : 531542 transactions
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0      0.0             0.0  
1  M2044282225             0.0             0.0      0.0             0.0  
2   C553264065             0.0             0.0      1.0             0.0  
3    C38997010         21182.0             0.0      1.0             0.0  
4  M1230701703             0.0             0.0      0.0             0.0  


In [4]:
from pyspark.sql.functions import col, when, sum, count

# Convertir en Spark DataFrame
df_spark = spark.createDataFrame(df)

print(f"✅ Spark DataFrame créé : {df_spark.count()} transactions")
df_spark.printSchema()

✅ Spark DataFrame créé : 531542 transactions
root
 |-- step: long (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: double (nullable = true)
 |-- isFlaggedFraud: double (nullable = true)



In [5]:
# Nettoyage
df_clean = df_spark.dropna().dropDuplicates()
print(f"Après nettoyage : {df_clean.count()} transactions")

# Transactions frauduleuses
df_fraud = df_clean.filter(col("isFraud") == 1)
print(f"Transactions frauduleuses : {df_fraud.count()}")

# Transactions normales
df_normal = df_clean.filter(col("isFraud") == 0)
print(f"Transactions normales : {df_normal.count()}")

Après nettoyage : 531541 transactions
Transactions frauduleuses : 235
Transactions normales : 531306


In [7]:
# Montant total par type de transaction
by_type = df_clean.groupBy("type").agg(
    count("*").alias("nb_transactions"),
    sum("amount").alias("montant_total")
).orderBy("montant_total", ascending=False)

display(by_type)

# Fraudes par type
fraud_by_type = df_fraud.groupBy("type").agg(
    count("*").alias("nb_fraudes"),
    sum("amount").alias("montant_fraude")
).orderBy("nb_fraudes", ascending=False)

fraud_by_type.show()

DataFrame[type: string, nb_transactions: bigint, montant_total: double]

+--------+----------+-------------------+
|    type|nb_fraudes|     montant_fraude|
+--------+----------+-------------------+
|CASH_OUT|       122|9.689487242999999E7|
|TRANSFER|       113|9.806699564999999E7|
+--------+----------+-------------------+



In [8]:
by_type = df_clean.groupBy("type").agg(
    count("*").alias("nb_transactions"),
    sum("amount").alias("montant_total")
).orderBy("montant_total", ascending=False)

by_type.show()

+--------+---------------+--------------------+
|    type|nb_transactions|       montant_total|
+--------+---------------+--------------------+
|CASH_OUT|         193499|3.579002700955984E10|
|TRANSFER|          43070|2.932053696098997...|
| CASH_IN|         116271|2.004428300233996...|
| PAYMENT|         174829|2.0052893657400064E9|
|   DEBIT|           3872| 2.485199287000005E7|
+--------+---------------+--------------------+



In [9]:
# Simuler le streaming par batch de 100 transactions
from pyspark.sql.functions import when

print("🔄 Simulation Streaming en temps réel...\n")

for i in range(0, 500, 100):
    batch = df_clean.limit(i+100).subtract(df_clean.limit(i))

    nb_fraudes = batch.filter(col("isFraud") == 1).count()
    nb_transactions = batch.count()

    print(f"Batch {i//100 + 1} → {nb_transactions} transactions | ⚠️ {nb_fraudes} fraudes détectées !")

print("\n✅ Streaming terminé !")

🔄 Simulation Streaming en temps réel...

Batch 1 → 100 transactions | ⚠️ 1 fraudes détectées !
Batch 2 → 100 transactions | ⚠️ 0 fraudes détectées !
Batch 3 → 100 transactions | ⚠️ 0 fraudes détectées !
Batch 4 → 100 transactions | ⚠️ 0 fraudes détectées !
Batch 5 → 100 transactions | ⚠️ 0 fraudes détectées !

✅ Streaming terminé !
